# nb05 — Embedding Model Evaluation

Pick a production retriever for the Kalisio RAG pipeline. Corpus is EN-only,
most queries come in FR, so cross-lingual retrieval has to be measured
empirically.

| Layer       | Query style                  | Tests                         |
|-------------|-------------------------------|-------------------------------|
| `A_symbol`  | API / component name          | exact identifier match        |
| `B_docs`    | Natural-language question     | semantic understanding        |
| `C_code`    | Concept → file                | cross-modal (NL → code)       |
| `negative`  | Out-of-scope question         | rejection                     |

Gold comes from `outputs/nb05_gold.json` (authored in `experiments/nb05_embedding_eval/gold_draft.json`).

In [1]:
import os, sys, gc, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "experiments" / "nb05_embedding_eval"))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[setup] device={DEVICE}", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
GOLD_PATH = OUTPUTS / "nb05_gold.json"


[setup] device=cuda NVIDIA GeForce RTX 3060 Ti


## 1. Load corpus and gold

In [2]:
from corpus_filter import scan_corpus
from corpus_filter.models import FilterConfig
from corpus_filter.profiles import build_js_vue_rag_profile
from chunking import chunk_files
from nb05_helpers import load_gold_validated, gold_summary

_base = build_js_vue_rag_profile()
cfg = FilterConfig(
    excluded_dirs=_base.excluded_dirs - {"docs"},
    excluded_extensions=_base.excluded_extensions,
    excluded_filenames=_base.excluded_filenames,
    excluded_patterns=_base.excluded_patterns,
    max_file_size=_base.max_file_size,
    max_line_length=_base.max_line_length,
    included_extensions={".md", ".js", ".mjs", ".vue", ".json"},
)
scan = scan_corpus(config=cfg)
chunks = chunk_files(scan.included)
chunk_texts   = [c["text"] for c in chunks]
chunk_sources = [c["metadata"]["source"] for c in chunks]
print(f"[corpus] files={len(scan.included)}  chunks={len(chunks)}")

queries = load_gold_validated(GOLD_PATH, chunk_sources)
query_en = [q.en for q in queries]
query_fr = [q.fr for q in queries]
print(f"[gold]   {gold_summary(queries)}")


[corpus] files=1244  chunks=9858
[gold]   {'A_symbol': 43, 'B_docs': 35, 'C_code': 23, 'negative': 6, 'total': 107}


In [3]:
print("Sample query per layer:\n")
seen = set()
for q in queries:
    if q.layer in seen:
        continue
    seen.add(q.layer)
    print(f"[{q.layer}] {q.id}")
    print(f"  EN: {q.en}")
    print(f"  FR: {q.fr}")
    print(f"  gold: {list(q.gold_sources)}\n")


Sample query per layer:

[A_symbol] A-001
  EN: addLayer function
  FR: fonction addLayer
  gold: ['kdk/docs/api/map/map-mixins.md', 'kdk/docs/api/map/globe-mixins.md']

[B_docs] B-001
  EN: How do I add a new layer to a map?
  FR: Comment ajouter une nouvelle couche à la carte ?
  gold: ['kdk/docs/api/map/map-mixins.md', 'kdk/docs/api/map/globe-mixins.md']

[C_code] C-001
  EN: Where is the addLayer logic implemented for the 2D map?
  FR: Où est implémentée la logique addLayer pour la carte 2D ?
  gold: ['kdk/core/client/mixins/mixin.service.js', 'kdk/map/client/mixins/map/mixin.base-map.js']

[negative] N-001
  EN: How do I integrate TensorFlow.js for ML predictions?
  FR: Comment intégrer TensorFlow.js pour des prédictions ML ?
  gold: []



## 2. Candidate recipes and truncation audit

In [4]:
from nb05_helpers import RECIPES, truncation_audit

display(pd.DataFrame([
    {
        "key": k,
        "model_id": r.model_id,
        "query_prefix": repr(r.query_prefix),
        "passage_prefix": repr(r.passage_prefix),
        "max_tokens": r.max_tokens,
        "matryoshka_dim": r.matryoshka_dim,
        "family": r.family,
    } for k, r in RECIPES.items()
]).set_index("key"))

audit_rows = [truncation_audit(r, chunk_texts) for r in RECIPES.values()]
df_audit = pd.DataFrame(audit_rows).set_index("model")
df_audit


,model_id,query_prefix,passage_prefix,max_tokens,matryoshka_dim,family
key,,,,,,
bge-m3,BAAI/bge-m3,'','',8192,NaN,multilingual-dense
e5-large,intfloat/multilingual-e5-large,'query: ','passage: ',512,NaN,multilingual-dense
nomic-v1.5,nomic-ai/nomic-embed-text-v1.5,'search_query: ','search_document: ',8192,768.0,english-dense
jina-code,jinaai/jina-embeddings-v2-base-code,'','',8192,NaN,code-dense
e5-large-instruct,intfloat/multilingual-e5-large-instruct,"'Instruct: Given a developer question, retrieve the relevant Kalisio documentation page or sourc...",'',512,NaN,multilingual-dense
arctic-l-v2,Snowflake/snowflake-arctic-embed-l-v2.0,'query: ','',8192,NaN,multilingual-dense


Token indices sequence length is longer than the specified maximum sequence length for this model (769 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (767 > 512). Running this sequence through the model will result in indexing errors


,max_tokens,n_chunks,n_truncated,truncation_rate,lost_fraction
model,,,,,
bge-m3,8192,9858,0,0.000000,0.000000
e5-large,512,9858,65,0.006594,0.010352
nomic-v1.5,8192,9858,0,0.000000,0.000000
jina-code,8192,9858,0,0.000000,0.000000
e5-large-instruct,512,9858,64,0.006492,0.010392
arctic-l-v2,8192,9858,0,0.000000,0.000000


## 3. Dense bake-off

In [5]:
from nb05_helpers import (
    encode_corpus, encode_queries, load_recipe_model,
    dense_rank, evaluate_ranks,
)

dense_ranks: dict[str, tuple[np.ndarray, np.ndarray]] = {}
dense_dfs: list[pd.DataFrame] = []
available_models: list[str] = []

for key, recipe in RECIPES.items():
    print(f"[dense] {key} -> loading")
    try:
        model = load_recipe_model(recipe)
    except Exception as exc:
        print(f"[dense] SKIP {key}: {type(exc).__name__}: {exc}")
        continue

    print(f"[dense] {key} -> encoding {len(chunks)} chunks")
    cv = encode_corpus(model, recipe, chunk_texts)
    en_v = encode_queries(model, recipe, query_en)
    fr_v = encode_queries(model, recipe, query_fr)

    en_r = dense_rank(en_v, cv)
    fr_r = dense_rank(fr_v, cv)
    dense_ranks[key] = (en_r, fr_r)
    available_models.append(key)

    dense_dfs.append(evaluate_ranks(en_r, queries, chunk_sources, language="en", approach=key))
    dense_dfs.append(evaluate_ranks(fr_r, queries, chunk_sources, language="fr", approach=key))

    del model, cv, en_v, fr_v
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

df_dense = pd.concat(dense_dfs, ignore_index=True)
print(f"\n[dense] available={available_models}")
df_dense.head()


[dense] bge-m3 -> loading
[dense] bge-m3 -> encoding 9858 chunks
[dense] e5-large -> loading


XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[dense] e5-large -> encoding 9858 chunks
[dense] nomic-v1.5 -> loading


<All keys matched successfully>


[dense] nomic-v1.5 -> encoding 9858 chunks
[dense] jina-code -> loading
[dense] SKIP jina-code: ImportError: cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/home/felix/miniconda3/envs/knowledge/lib/python3.11/site-packages/transformers/pytorch_utils.py)
[dense] e5-large-instruct -> loading
[dense] e5-large-instruct -> encoding 9858 chunks
[dense] arctic-l-v2 -> loading
[dense] arctic-l-v2 -> encoding 9858 chunks

[dense] available=['bge-m3', 'e5-large', 'nomic-v1.5', 'e5-large-instruct', 'arctic-l-v2']


,approach,language,layer,query_id,is_negative,hit@k,recall@k,mrr
0,bge-m3,en,A_symbol,A-001,False,0,0.0,0.015152
1,bge-m3,en,A_symbol,A-002,False,0,0.0,0.016667
2,bge-m3,en,A_symbol,A-003,False,1,1.0,1.000000
3,bge-m3,en,A_symbol,A-004,False,0,0.0,0.008000
4,bge-m3,en,A_symbol,A-005,False,0,0.0,0.004739


## 4. BM25 baseline

In [6]:
from nb05_helpers import bm25_rank

bm25_en_ranks = bm25_rank(chunk_texts, query_en)
bm25_fr_ranks = bm25_rank(chunk_texts, query_fr)
df_bm25 = pd.concat([
    evaluate_ranks(bm25_en_ranks, queries, chunk_sources, language="en", approach="bm25"),
    evaluate_ranks(bm25_fr_ranks, queries, chunk_sources, language="fr", approach="bm25"),
], ignore_index=True)
df_bm25.groupby(["layer", "language"])["hit@k"].mean().unstack("language").round(3)


language,en,fr
layer,,
A_symbol,0.837,0.814
B_docs,0.371,0.086
C_code,0.304,0.087
negative,0.500,1.000


## 5. Hybrid (Dense + BM25 via RRF)

In [7]:
from nb05_helpers import rrf_fuse

if not available_models:
    print("[hybrid] no dense models available, skipping")
    df_hybrid = pd.DataFrame()
else:
    leader_key = (
        df_dense.groupby("approach")["hit@k"].mean().idxmax()
    )
    print(f"[hybrid] leader = {leader_key}")
    en_leader, fr_leader = dense_ranks[leader_key]

    hybrid_dfs = []
    for k_rrf in (30, 60, 90):
        en_fused = rrf_fuse(en_leader, bm25_en_ranks, k_rrf=k_rrf)
        fr_fused = rrf_fuse(fr_leader, bm25_fr_ranks, k_rrf=k_rrf)
        approach = f"hybrid_k{k_rrf}"
        hybrid_dfs.append(evaluate_ranks(en_fused, queries, chunk_sources, language="en", approach=approach))
        hybrid_dfs.append(evaluate_ranks(fr_fused, queries, chunk_sources, language="fr", approach=approach))
    df_hybrid = pd.concat(hybrid_dfs, ignore_index=True)

df_hybrid.groupby(["approach", "layer"])["hit@k"].mean().unstack("layer").round(3)


[hybrid] leader = arctic-l-v2


layer,A_symbol,B_docs,C_code,negative
approach,,,,
hybrid_k30,0.837,0.471,0.543,0.583
hybrid_k60,0.837,0.486,0.543,0.583
hybrid_k90,0.837,0.457,0.543,0.583


## 6. Cross-encoder reranker (Layer B + C only)

In [8]:
from nb05_helpers import load_reranker, rerank_topk

K_CAND = 20
df_rerank = pd.DataFrame()
if df_hybrid.empty:
    print("[reranker] skipped (no hybrid baseline)")
else:
    try:
        reranker = load_reranker()
    except Exception as exc:
        print(f"[reranker] unavailable: {type(exc).__name__}: {exc}")
        reranker = None

    if reranker is not None:
        best_k = (
            df_hybrid.groupby("approach")["hit@k"].mean().idxmax().split("_k")[1]
        )
        best_k = int(best_k)
        print(f"[reranker] reranking on top of hybrid_k{best_k}")
        en_leader, fr_leader = dense_ranks[leader_key]
        en_fused = rrf_fuse(en_leader, bm25_en_ranks, k_rrf=best_k)
        fr_fused = rrf_fuse(fr_leader, bm25_fr_ranks, k_rrf=best_k)

        rerank_eligible = [i for i, q in enumerate(queries) if q.layer in {"B_docs", "C_code"}]

        def rerank_one(fused: np.ndarray, q_text: str, qi: int) -> np.ndarray:
            cand = fused[qi, :K_CAND].tolist()
            order = rerank_topk(reranker, q_text, [chunks[c]["text"] for c in cand])
            reranked = [cand[o] for o in order]
            return np.array(reranked + fused[qi, K_CAND:].tolist())

        en_reranked = en_fused.copy()
        fr_reranked = fr_fused.copy()
        for qi in rerank_eligible:
            en_reranked[qi] = rerank_one(en_fused, queries[qi].en, qi)
            fr_reranked[qi] = rerank_one(fr_fused, queries[qi].fr, qi)

        df_rerank = pd.concat([
            evaluate_ranks(en_reranked, queries, chunk_sources, language="en", approach=f"hybrid_k{best_k}+rerank"),
            evaluate_ranks(fr_reranked, queries, chunk_sources, language="fr", approach=f"hybrid_k{best_k}+rerank"),
        ], ignore_index=True)

        del reranker
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if not df_rerank.empty:
    display(df_rerank[df_rerank["layer"].isin({"B_docs", "C_code"})]
            .groupby(["layer", "language"])["hit@k"].mean().unstack("language").round(3))


[reranker] reranking on top of hybrid_k60


language,en,fr
layer,,
B_docs,0.714,0.457
C_code,0.739,0.522


## 7. Per-layer leaderboard

In [9]:
from nb05_helpers import leaderboard

df_all = pd.concat([df_dense, df_bm25, df_hybrid, df_rerank], ignore_index=True)

for layer in ("A_symbol", "B_docs", "C_code", "negative"):
    print(f"\n=== {layer} ===")
    display(leaderboard(df_all, layer=layer))



=== A_symbol ===


language,en,fr,mean
approach,,,
nomic-v1.5,0.860,0.837,0.848
e5-large,0.837,0.860,0.848
hybrid_k60+rerank,0.837,0.837,0.837
hybrid_k90,0.837,0.837,0.837
hybrid_k60,0.837,0.837,0.837
hybrid_k30,0.837,0.837,0.837
bm25,0.837,0.814,0.825
arctic-l-v2,0.814,0.837,0.825
e5-large-instruct,0.814,0.814,0.814



=== B_docs ===


language,en,fr,mean
approach,,,
hybrid_k60+rerank,0.714,0.457,0.586
arctic-l-v2,0.543,0.514,0.528
e5-large,0.571,0.429,0.500
bge-m3,0.543,0.429,0.486
hybrid_k60,0.571,0.400,0.486
hybrid_k30,0.571,0.371,0.471
hybrid_k90,0.571,0.343,0.457
e5-large-instruct,0.543,0.229,0.386
nomic-v1.5,0.457,0.229,0.343



=== C_code ===


language,en,fr,mean
approach,,,
hybrid_k60+rerank,0.739,0.522,0.631
arctic-l-v2,0.696,0.565,0.630
e5-large,0.609,0.565,0.587
bge-m3,0.609,0.522,0.566
hybrid_k60,0.565,0.522,0.544
hybrid_k30,0.565,0.522,0.544
hybrid_k90,0.609,0.478,0.544
nomic-v1.5,0.739,0.217,0.478
e5-large-instruct,0.478,0.217,0.348



=== negative ===


language,en,fr,mean
approach,,,
e5-large-instruct,1.000,1.000,1.000
bm25,0.500,1.000,0.750
bge-m3,0.667,0.667,0.667
arctic-l-v2,0.667,0.667,0.667
nomic-v1.5,0.500,0.833,0.666
e5-large,0.500,0.667,0.584
hybrid_k30,0.500,0.667,0.584
hybrid_k60,0.500,0.667,0.584
hybrid_k60+rerank,0.500,0.667,0.584


## 8. Cost profile

In [10]:
from nb05_helpers import measure_throughput, index_size_bytes, query_latency_ms

cost_rows = []
sample_q = query_en[: min(40, len(query_en))]
sample_chunks = chunk_texts[: min(512, len(chunk_texts))]

for key in available_models:
    recipe = RECIPES[key]
    try:
        model = load_recipe_model(recipe)
    except Exception:
        continue

    thr = measure_throughput(model, recipe, sample_chunks)
    full_cv = encode_corpus(model, recipe, chunk_texts)
    encode_q = lambda q, m=model, r=recipe: encode_queries(m, r, [q])
    lat = query_latency_ms(encode_q, full_cv, sample_q)

    cost_rows.append({
        "model": key,
        "dim": int(full_cv.shape[1]),
        "chunks_per_sec": round(thr["chunks_per_sec"], 1),
        "query_mean_ms": round(lat["mean_ms"], 1),
        "query_p95_ms":  round(lat["p95_ms"], 1),
        "index_mb": round(index_size_bytes(len(chunks), full_cv.shape[1]) / (1024 ** 2), 1),
    })
    del model, full_cv
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

df_cost = pd.DataFrame(cost_rows).set_index("model")
df_cost


XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
<All keys matched successfully>


,dim,chunks_per_sec,query_mean_ms,query_p95_ms,index_mb
model,,,,,
bge-m3,1024,52.9,11.0,11.3,38.5
e5-large,1024,51.1,11.1,11.2,38.5
nomic-v1.5,768,130.7,8.5,8.9,28.9
e5-large-instruct,1024,169.8,39.5,39.7,38.5
arctic-l-v2,1024,52.5,11.1,11.4,38.5


## 9. Per-layer winners

In [ ]:
from nb05_helpers import per_layer_summary

per_layer = per_layer_summary(df_all)
winners = {layer: per_layer[layer].idxmax() for layer in ("A_symbol", "B_docs", "C_code")}
print("Winners:", winners)
display(per_layer)


## 10. Persist results

In [12]:
df_all.to_json(OUTPUTS / "nb05_results.json", orient="records", indent=2)
df_cost.to_json(OUTPUTS / "nb05_cost.json", orient="index", indent=2)
df_audit.to_json(OUTPUTS / "nb05_truncation.json", orient="index", indent=2)
print("[save] outputs:")
for p in ("nb05_results.json", "nb05_cost.json", "nb05_truncation.json", "nb05_recommendation.md"):
    print(" -", OUTPUTS / p)


[save] outputs:
 - /home/felix/kalisio/knowledge/outputs/nb05_results.json
 - /home/felix/kalisio/knowledge/outputs/nb05_cost.json
 - /home/felix/kalisio/knowledge/outputs/nb05_truncation.json
 - /home/felix/kalisio/knowledge/outputs/nb05_recommendation.md


## 11. FR-first view

In [13]:
# FR-only per-layer leaderboard
fr = df_all[df_all["language"] == "fr"]
fr_per_layer = (
    fr.groupby(["approach", "layer"])["hit@k"].mean()
    .unstack("layer").round(3)
)
# Order columns deterministically; layers may be missing if no queries
ordered_cols = [c for c in ("A_symbol", "B_docs", "C_code", "negative") if c in fr_per_layer.columns]
fr_per_layer = fr_per_layer[ordered_cols]
fr_per_layer["B+C mean"] = fr_per_layer[[c for c in ("B_docs", "C_code") if c in fr_per_layer.columns]].mean(axis=1).round(3)
fr_per_layer = fr_per_layer.sort_values("B+C mean", ascending=False)
print("=== FR-only hit@5 by approach × layer (sorted by B+C mean) ===\n")
display(fr_per_layer)


=== FR-only hit@5 by approach × layer (sorted by B+C mean) ===



layer,A_symbol,B_docs,C_code,negative,B+C mean
approach,,,,,
arctic-l-v2,0.837,0.514,0.565,0.667,0.540
e5-large,0.860,0.429,0.565,0.667,0.497
hybrid_k60+rerank,0.837,0.457,0.522,0.667,0.490
bge-m3,0.744,0.429,0.522,0.667,0.476
hybrid_k60,0.837,0.400,0.522,0.667,0.461
hybrid_k30,0.837,0.371,0.522,0.667,0.446
hybrid_k90,0.837,0.343,0.478,0.667,0.410
e5-large-instruct,0.814,0.229,0.217,1.000,0.223
nomic-v1.5,0.837,0.229,0.217,0.833,0.223


In [14]:
# Cost-accuracy frontier on FR mean (B+C layers — where the hard work is)
if not df_cost.empty:
    fr_score = (
        df_all[(df_all["language"] == "fr") & (df_all["layer"].isin(["B_docs", "C_code"]))]
        .groupby("approach")["hit@k"].mean()
    )
    frontier = (
        df_cost.assign(fr_BC_hit5=df_cost.index.map(fr_score).round(3))
        .dropna(subset=["fr_BC_hit5"])
        .sort_values("fr_BC_hit5", ascending=False)
        [["dim", "chunks_per_sec", "query_mean_ms", "index_mb", "fr_BC_hit5"]]
    )
    print("=== Dense models: FR B+C hit@5 vs cost ===\n")
    display(frontier)
else:
    frontier = pd.DataFrame()
    print("(no cost data — dense models did not load)")


=== Dense models: FR B+C hit@5 vs cost ===



,dim,chunks_per_sec,query_mean_ms,index_mb,fr_BC_hit5
model,,,,,
arctic-l-v2,1024,52.5,11.1,38.5,0.534
e5-large,1024,51.1,11.1,38.5,0.483
bge-m3,1024,52.9,11.0,38.5,0.466
nomic-v1.5,768,130.7,8.5,28.9,0.224
e5-large-instruct,1024,169.8,39.5,38.5,0.224


In [15]:
def _layer_mean(df, lang, layers):
    sub = df[(df["language"] == lang) & (df["layer"].isin(layers))]
    return sub.groupby("approach")["hit@k"].mean()

# FR-weighted score: 0.6 FR(B+C) + 0.2 FR(A) + 0.2 EN(B+C)
fr_bc  = _layer_mean(df_all, "fr", ["B_docs", "C_code"])
fr_a   = _layer_mean(df_all, "fr", ["A_symbol"])
en_bc  = _layer_mean(df_all, "en", ["B_docs", "C_code"])
neg_fr = _layer_mean(df_all, "fr", ["negative"])

scored = (0.6 * fr_bc + 0.2 * fr_a + 0.2 * en_bc).sort_values(ascending=False).round(3)
top_table = pd.DataFrame({
    "weighted_score": scored,
    "FR_B+C": fr_bc.round(3),
    "FR_A":   fr_a.round(3),
    "EN_B+C": en_bc.round(3),
    "FR_neg": neg_fr.round(3),
}).loc[scored.index]
display(top_table.head(10))
print(f"Top-3: {scored.head(3).index.tolist()}")


=== Top approaches by FR-weighted score ===



,weighted_score,FR_B+C,FR_A,EN_B+C,FR_neg
approach,,,,,
arctic-l-v2,0.609,0.534,0.837,0.603,0.667
hybrid_k60+rerank,0.602,0.483,0.837,0.724,0.667
e5-large,0.579,0.483,0.860,0.586,0.667
hybrid_k60,0.550,0.448,0.837,0.569,0.667
bge-m3,0.542,0.466,0.744,0.569,0.667
hybrid_k30,0.540,0.431,0.837,0.569,0.667
hybrid_k90,0.523,0.397,0.837,0.586,0.667
nomic-v1.5,0.416,0.224,0.837,0.569,0.833
e5-large-instruct,0.401,0.224,0.814,0.517,1.000



Top-3: ['arctic-l-v2', 'hybrid_k60+rerank', 'e5-large']
